In [1]:
import pandas as pd

caminho = "../data/raw/vacinabr-dados-por-municipio.csv"

usecols = [
    "ibge6", "uf", "ano", "imunizante", "faixa_etaria", "doenca",
    "populacao", "qt_dose", "cobertura", "meta", "meta_atingida",
    "abandono", "homog_vacinas", "reforco", "ultima_dose",
]

vacinas_mvp = [
    "BCG",
    "Poliomielite inativada",
    "Pentavalente",
    "Tríplice Viral",
    "Hepatite B (0 a 30 dias)",
]

chunks_filtrados = []

for chunk in pd.read_csv(caminho, usecols=usecols, chunksize=500_000):
    filtro = (
        chunk["imunizante"].isin(vacinas_mvp)
        & (chunk["ultima_dose"] == True)
        & (chunk["reforco"] == 0)
        & (chunk["ano"] >= 2014)
        & (chunk["ano"] <= 2024)
    )
    chunks_filtrados.append(chunk[filtro])

df = pd.concat(chunks_filtrados, ignore_index=True)

print(f"Total de linhas após filtro: {len(df)}")

grupos = df.groupby(["ibge6", "ano", "imunizante"]).size()
duplicados = grupos[grupos > 1]

print(f"Combinações (ibge6, ano, imunizante) com mais de 1 linha: {len(duplicados)}")

if len(duplicados) > 0:
    ibge6_ex, ano_ex, imunizante_ex = duplicados.index[0]
    exemplo = df[
        (df["ibge6"] == ibge6_ex)
        & (df["ano"] == ano_ex)
        & (df["imunizante"] == imunizante_ex)
    ].head(5)
    print()
    print(f"Exemplo de grupo duplicado: ibge6={ibge6_ex}, ano={ano_ex}, imunizante={imunizante_ex}")
    print(exemplo[["ibge6", "ano", "imunizante", "doenca"]])

C:\Users\Joao Vitor\AppData\Local\Temp\ipykernel_29700\3825656031.py:21: DtypeWarning: Columns (5,8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(caminho, usecols=usecols, chunksize=500_000):


C:\Users\Joao Vitor\AppData\Local\Temp\ipykernel_29700\3825656031.py:21: DtypeWarning: Columns (5,8) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(caminho, usecols=usecols, chunksize=500_000):


Total de linhas após filtro: 221500


Combinações (ibge6, ano, imunizante) com mais de 1 linha: 0


In [2]:
print(df["ano"].value_counts().sort_index())
print()
print(df["imunizante"].value_counts())

ano
2014    22183
2015    22193
2016    22125
2017    22171
2018    22144
2019    22096
2020    22060
2021    22090
2022    22158
2023    22280
Name: count, dtype: int64

imunizante
Pentavalente              55676
Tríplice Viral            55675
Poliomielite inativada    55657
BCG                       54492
Name: count, dtype: int64
